# 02 · User-Prioritized Embedding
**Stage 1 of LPS — Agency-Weighted Representation**

Implements:

$$v_t = \alpha \cdot \text{enc}(U_t) + (1 - \alpha) \cdot \text{enc}(M_t)$$

where $\alpha \in [0.5, 1.0]$ is the **Agency Coefficient** that amplifies the
user's own words relative to model-generated verbosity, directly countering
**Model Verbosity Bias**.


In [ ]:

import sys
sys.path.append("../src")

import numpy as np
import json, yaml
from tqdm import tqdm
from embeddings import UserPrioritizedEmbedder

with open("../configs/default.yaml") as f:
    cfg = yaml.safe_load(f)

with open("../data/cohort.json") as f:
    cohort = json.load(f)

print(f"Loaded {len(cohort)} users")


In [ ]:

# ── Initialise embedder ───────────────────────────────────────────────
embedder = UserPrioritizedEmbedder(
    model_name="sentence-transformers/all-mpnet-base-v2",  # swap for text-embedding-3-large via API
    alpha=cfg["embedding"]["alpha"],
    batch_size=cfg["embedding"]["batch_size"],
)
print(f"Agency Coefficient α = {embedder.alpha}")


In [ ]:

# ── Embed a single user's history ────────────────────────────────────
uid = list(cohort.keys())[0]
user_data = cohort[uid]

interactions = []
timestamps   = []
for conv in user_data["history"]:
    for pair in conv["pairs"]:
        interactions.append((pair["user"], pair["model"]))
        timestamps.append(conv["timestamp"])

print(f"User {uid[:12]}... — {len(interactions)} interactions")

embeddings = embedder.embed_interactions(interactions)
print(f"Embedding shape: {embeddings.shape}")
print(f"Sample norm (should ≈ 1.0): {np.linalg.norm(embeddings[0]):.4f}")


In [ ]:

# ── Visualise effect of α on embedding space ─────────────────────────
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
alphas = [0.5, 0.75, 1.0]

for ax, alpha in zip(axes, alphas):
    emb_test = UserPrioritizedEmbedder(alpha=alpha)
    embs = emb_test.embed_interactions(interactions[:50])
    pca  = PCA(n_components=2)
    pts  = pca.fit_transform(embs)
    ax.scatter(pts[:, 0], pts[:, 1], alpha=0.6, s=15)
    ax.set_title(f"α = {alpha}")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")

plt.suptitle("PCA of User-Prioritized Embeddings at different α values", fontsize=13)
plt.tight_layout()
plt.savefig("../data/alpha_pca.png", dpi=120)
plt.show()
print("Plot saved to ../data/alpha_pca.png")


In [ ]:

# ── Embed and save all users ──────────────────────────────────────────
import pickle

all_embeddings = {}
for uid, user_data in tqdm(cohort.items(), desc="Embedding users"):
    interactions_u, timestamps_u = [], []
    for conv in user_data["history"]:
        for pair in conv["pairs"]:
            interactions_u.append((pair["user"], pair["model"]))
            timestamps_u.append(conv["timestamp"])
    embs = embedder.embed_interactions(interactions_u)
    all_embeddings[uid] = {
        "embeddings":   embs,
        "timestamps":   timestamps_u,
        "interactions": [{"user": u, "model": m} for u, m in interactions_u],
    }

with open("../data/embeddings.pkl", "wb") as f:
    pickle.dump(all_embeddings, f)

print(f"Saved embeddings for {len(all_embeddings)} users")
print("\n✓ Notebook 02 complete — proceed to Notebook 03 (ELC) or 04 (Graph-RAG)")
